# Notebook 12 — Interpretability and Model Debugging

    ## Learning objectives

    - Inspect activations, attention, logits, and gradients with hooks
- Use probes and interventions without overstating causality
- Connect mechanistic evidence to behavioral evaluation

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

# VS Code's Colab extension can attach to a Colab kernel before `google.colab`
# has been imported, so checking only sys.modules produces a false negative.
try:
    HAS_GOOGLE_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:  # The parent `google` namespace is absent locally.
    HAS_GOOGLE_COLAB = False
IN_COLAB = HAS_GOOGLE_COLAB or bool(os.getenv("COLAB_RELEASE_TAG")) or bool(os.getenv("COLAB_GPU"))
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from the Colab Secrets UI without displaying it. Create a
# secret named exactly HF_TOKEN and enable notebook access with its toggle.
token = os.getenv("HF_TOKEN")
token_error = None
if IN_COLAB and not token:
    from google.colab import userdata
    try:
        token = userdata.get("HF_TOKEN")
    except Exception as exc:
        token_error = type(exc).__name__
elif not IN_COLAB:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass
    token = os.getenv("HF_TOKEN")

# `HF_TOKEN` is the canonical huggingface_hub environment variable.
if token:
    os.environ["HF_TOKEN"] = token

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Colab runtime detected:", IN_COLAB)
print("Hugging Face token configured:", bool(os.getenv("HF_TOKEN")))
if IN_COLAB and not token:
    print("HF_TOKEN is unavailable. In Colab, open the key icon (Secrets), add HF_TOKEN, ")
    print("enable its Notebook access toggle, and rerun this cell. Public models still work.")
    if token_error:
        print("Colab secret lookup status:", token_error)


## 12.1 Observation versus explanation

Hidden states, attention weights, gradients, and token probabilities are measurements of a particular forward pass. Attention weight is not automatically feature importance or causal explanation. A linear probe shows information is decodable, not that the model uses it. Begin with a behavioral phenomenon and falsifiable hypothesis, choose controls, and preserve exact prompt/tokenizer/model revisions. Aggregate plots can erase token-, head-, layer-, or example-specific behavior.


In [ ]:
import torch
from torch import nn
model=nn.Sequential(nn.Linear(4,8),nn.ReLU(),nn.Linear(8,3)); captured={}
h=model[0].register_forward_hook(lambda m,i,o: captured.update(hidden=o.detach())); out=model(torch.randn(2,4)); h.remove(); print(out.shape,captured["hidden"].shape)


## 12.2 Hooks and logit lens

PyTorch forward hooks capture module inputs or outputs; remove them after use and avoid mutation. Requesting hidden states or attentions increases memory and can change optimized kernels. A logit lens applies the final normalization and unembedding to intermediate residual states to inspect evolving token evidence, but intermediate representations were not necessarily trained to be directly decoded. Compare with tuned lenses or interventions and include random or shuffled controls.


In [ ]:
hidden=torch.randn(2,5,8); unembed=torch.randn(8,11); probs=(hidden@unembed).softmax(-1); print(probs.shape,probs.argmax(-1))


## 12.3 Probes, attribution, and patching

Probes range from linear classifiers to expressive models; high-capacity probes can learn the task rather than reveal representation. Use train/validation splits, balanced baselines, selectivity controls, and layer sweeps. Gradient attribution is local and sensitive to saturation or baselines. Activation patching replaces a component between clean and corrupted runs and measures causal restoration, but intervention location, metric, and distribution matter. Multiple components may be redundant.


In [ ]:
clean=torch.tensor([1.,2.,3.]); corrupt=torch.tensor([1.,-2.,3.]); patched=corrupt.clone(); patched[1]=clean[1]
metric=lambda x:x.sum(); print(metric(clean),metric(corrupt),metric(patched))


## 12.4 Features and responsible claims

Neuron inspection often finds polysemantic units. Sparse autoencoders attempt to decompose activations into sparse learned features but depend on dataset, layer, width, sparsity, and interpretation method. Automated labels need human and causal validation. Interpretability can debug tokenization, masks, template boundaries, memorization, and regressions; it is not a safety certificate. Release claims with methods, prompts, controls, uncertainty, counterexamples, and code sufficient to reproduce the intervention.


In [ ]:
labels=torch.tensor([0,0,1,1]); feature=torch.tensor([-.9,-.7,.8,.6]); print("threshold probe",((feature>0).long()==labels).float().mean().item())


## Reference workflow and evidence standard

Treat the notebook as an experiment, not a recipe. State the question, freeze inputs and
success criteria, establish the simplest baseline, change one material factor, and retain raw
outputs needed to diagnose failures. Record model, tokenizer, template, data and code revisions;
hardware and dtype; random seeds; generation or optimization configuration; token counts;
latency and memory; and results by meaningful slice. A demonstration that runs is evidence of
plumbing, not evidence of general capability.

Test boundaries as well as the happy path: empty and maximum-length inputs, malformed records,
multilingual or code text, unavailable dependencies, cancellation, and adversarial content.
Keep credentials in environment or Colab Secrets and never serialize them with artifacts. Pin
remote revisions, review licenses and custom code, validate saved artifacts in a fresh process,
and prefer deterministic validators wherever outputs can be checked mechanically.

Before applying the technique, compare it with prompting, retrieval, a smaller model, or no
model. Report quality together with compute, storage, latency, and operational complexity. Use
held-out data and paired comparisons, disclose uncertainty and negative results, and define a
rollback path. These practices connect low-level understanding to reliable application work.

A useful completion checklist asks four separate questions. Is the mathematical contract clear
enough to predict shapes, masks, reductions, and failure cases? Does the implementation reproduce
a tiny hand-worked or deterministic reference? Does the measured result survive a held-out set,
relevant slices, and an ablation against a simpler baseline? Can another person reload the exact
artifacts and reconstruct the claim from the manifest? Passing only the first two establishes a
tutorial demonstration; passing all four supports an engineering decision. When a result fails,
preserve the counterexample and update the test suite before changing the implementation.

Finally, separate correctness, capability, efficiency, and safety conclusions. A correct
implementation may have weak capability; a capable prototype may be too costly or unsafe to
deploy. Name the population to which each conclusion applies and avoid converting a single
metric into a universal ranking. Track assumptions beside results, especially tokenizer and
template compatibility, data rights, access-control boundaries, and hardware-specific behavior.
Leave exercises with an executable acceptance criterion, a baseline result, and a short written
interpretation. That combination turns exploratory code into cumulative course evidence that can
be revisited when libraries, model families, or deployment engines change.


## 12.5 A controlled activation-patching experiment

Begin with clean and corrupted prompts whose behavioral difference is stable. Cache candidate activations from the clean run, patch one component into the corrupted run, and measure restoration using a prespecified logit-difference or task metric. Include same-condition patches, random activations, unrelated token positions, and multiple examples. Normalize restoration carefully because a small clean-corrupt denominator makes ratios unstable. Sweep layers, positions, and components with multiple-comparison awareness. Patching supports a causal claim about the intervention and metric under these inputs; it does not prove a human-readable concept is localized there.


In [ ]:
clean_score=torch.tensor(3.0); corrupt_score=torch.tensor(-1.0); patched={"early":torch.tensor(-.8),"middle":torch.tensor(2.2),"late":torch.tensor(.3)}
for site,score in patched.items():
 restoration=((score-corrupt_score)/(clean_score-corrupt_score)).item(); print(site,restoration)
print("control patch restoration",float((torch.tensor(-.9)-corrupt_score)/(clean_score-corrupt_score)))


## 12.6 Hooks without leaks or behavioral changes

Forward and backward hooks are powerful observability tools but easy to misuse. Store detached summaries instead of full graph-connected activations, remove handles in a `finally` block, avoid in-place edits, and confirm instrumented and uninstrumented outputs match. Capturing all layers for long sequences can exhaust memory. Prefer selective modules, tokens, batches, and streaming statistics. Module hooks may behave unexpectedly with reused modules or compiled graphs, so identify calls and test the exact runtime. Treat interpretability artifacts as sensitive when activations can reveal input content, and record model, tokenizer, prompt, layer, hook point, and aggregation.


In [ ]:
captured=[]
handle=model[0].register_forward_hook(lambda module,args,out: captured.append({"mean":out.detach().mean().item(),"shape":tuple(out.shape)}))
try:
 probe=torch.randn(3,4); instrumented=model(probe)
finally:
 handle.remove()
plain=model(probe); torch.testing.assert_close(instrumented,plain); print(captured,"remaining hooks",len(model[0]._forward_hooks))


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [A Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html)
- [Tracing the mechanisms of language model factual recall](https://arxiv.org/abs/2202.05262)


## Exercises

    1. Capture and remove hooks safely.
2. Design an activation-patching control.
3. Critique an attention-as-explanation claim.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
